# Appendix figures — nearest-neighbor Ising model

Reproduces the nearest-neighbor Ising appendix figures from the small summary dicts committed in
this folder (`hld_dict_for_last_figs.jld2`, `all_temp_level_dict.jld2`). The published PDFs were
composited in Inkscape; this reproduces the underlying panels.

- **`nn_ising_kappa_C.pdf` panels a–d** (Appendix C) — per-energy-level reversed D_KL(q̂_{τ=1}‖p_T)
  and its τ-derivative at τ=1, for a low-T (T=2.3) and a high-T (T=4.0) single experiment
  (M=54, 4×4). Positive derivative ⇒ lower τ; negative ⇒ raise τ. Data: `hld_dict_for_last_figs.jld2`.
- **`supp_fig_working-new.pdf`** (Appendix D) — mean-over-replicates D_KL(p_D‖p_T)∩Λ_4 and
  D_KL(q̂‖p_T)∩Λ_4 vs training-set size M, for several ground-truth temperatures (inset: the
  same without the Λ_4 restriction). Data: `all_temp_level_dict.jld2` (assembled from the
  per-temperature per-level D_KL sweeps, T = 2.3, 2.4, …, 4.4).

The per-level D_KL data in `all_temp_level_dict.jld2` was generated by an independent
re-sampling/re-fitting sweep (`dkl_per_lev_M_T.jl` — see `scripts/build_supplemental_dicts.jl`),
NOT from the raw sweeps on Hugging Face, so it is committed here directly.

> **Not reproduced in this notebook yet:**
> - **Panel (e)** of `nn_ising_kappa_C` (C(τ=1), κ(τ=1) vs ground-truth T) — recomputed in the
>   source from the raw fitted models (per-state heat-/cross-capacity), not from a committed summary.
> - **`fig4-working-latest.pdf`** (Appendix D) — per-state D_KL-bias panels; needs the `fig4_2` +
>   `make_hist_edges` helpers ported.


In [ ]:
using DrWatson
@quickactivate "temp-tune"

# PyPlot + Printf must load before the src include (macros expand at include time).
using JLD2, PyPlot, Statistics, Printf

include(srcdir("matplotlib_helpers.jl"))   # set_size_for_paper, setup_notebook_for_paper, using_Py

setup_notebook_for_paper()
using_Py(true)

pe = pyimport("matplotlib.patheffects")    # path-effects used by supp_fig

In [ ]:
# --- nn_ising_kappa_C panels a–d (ported from proteins/temp_analysis/ml_fitting_3.ipynb) ---

"""
Panels a–d of appendix figure `nn_ising_kappa_C.pdf`: per-energy-level reversed
D_KL(q̂_{τ=1}||p_T) and its τ-derivative at τ=1, for the low-T (T=2.3) and high-T (T=4.0)
single experiments stored in `hld[:low]` / `hld[:high]`. A positive derivative ⇒ lower τ;
negative ⇒ raise τ. `ax` is a length-4 (2×2) axes container.
"""
function plot_dkl_et_al(ax, hld)
    lvl_resort = hld[:low][:lvl_resort]
    xx = collect(0:length(lvl_resort) - 1)
    iiddxxx = 2:10   # excited levels shown in the paper

    ax[1].bar(xx[iiddxxx], hld[:low][:dkl_q_p_per_lev][iiddxxx],        color="k")
    ax[2].bar(xx[iiddxxx], hld[:low][:DdklDtau_q_p_per_level][iiddxxx], color="k")
    ax[3].bar(xx[iiddxxx], hld[:high][:dkl_q_p_per_lev][iiddxxx],       color="k")
    ax[4].bar(xx[iiddxxx], hld[:high][:DdklDtau_q_p_per_level][iiddxxx], color="k")

    ax[1].set_ylabel(L"D_{KL}(\hat{q}_{\tau=1} || p_T)" * "\nper energy level")
    ax[2].set_ylabel(L"\frac{\partial}{\partial\tau} \left[D_{KL}(\hat{q}_{\tau} || p_T)\right]_{\tau=1}" * "\nper energy level")
    ax[2].set_xticks(xx[iiddxxx])

    ax[1].set_title("low T experiment", fontsize=8)
    ax[3].set_title("high T experiment", fontsize=8)
    ax[2].set_xlabel("energy level index")
    ax[4].set_xlabel("energy level index")

    ax[4].set_ylim(-0.45, 0.8)
    ax[3].set_ylim(-0.02, 0.35)
    ax[1].sharey(ax[3]); ax[2].sharey(ax[4])
    ax[3].tick_params(axis="y", labelleft=false)
    ax[4].tick_params(axis="y", labelleft=false)
    return nothing
end

# --- supp_fig_working-new.pdf (Appendix D): mean D_KL vs M for several T ---

"Map temperature t to a color by normalizing within [min,max] of `all_t`."
get_c(c_map_, all_t, t) = c_map_(Float64((t - minimum(all_t)) / (maximum(all_t) - minimum(all_t))))

"""
Reproduce `supp_fig_working-new.pdf`: mean over reps of D_KL(p_D‖p_T)∩Λ_4 and
D_KL(q̂‖p_T)∩Λ_4 vs training-set size M, for several ground-truth temperatures
(inset: the same D_KLs without the Λ_4 level restriction). Uses globals
`all_temp_level_dict`, `all_tts`, `Mss`, `pe`, `fig`.
"""
function fig4_3(ax; inset_loc=(0.6, 0.6, 0.3, 0.3), cb_loc=[0.61, 0.35, 0.27, 0.025],
                axinlegloc=(0.25, 0.5), legloc=(0.6, 0.45))
    clrs  = [matplotlib.colors.to_rgba("darkolivegreen", alpha=aaa) for aaa in collect(0:0.001:1)]
    clrs2 = [matplotlib.colors.to_rgba("steelblue",      alpha=aaa) for aaa in collect(0:0.001:1)]
    cc  = matplotlib.colors.LinearSegmentedColormap.from_list("alpha_cmap",  clrs,  N=1001)
    cc2 = matplotlib.colors.LinearSegmentedColormap.from_list("alpha_cmap2", clrs2, N=1001)
    cc3 = get_cmap("Greys")

    axin = ax.inset_axes(inset_loc)
    axin.tick_params(axis="both", which="both", top=true, labeltop=true, bottom=false, right=true,
        left=false, labelright=true, labelleft=false, labelbottom=false, pad=1)

    lvl_id_1 = 4
    lvl_id_2 = 4
    allttsminor = all_tts[[1, 3, 5, 7]]
    allttsminorforc = vcat(0.8 * minimum(allttsminor), allttsminor, 1.06 * maximum(allttsminor))

    for (iil, t) in enumerate(allttsminor)
        mss_id = 1:13
        y  = vec(mean(sum(all_temp_level_dict[t][:all_dkl_per_lev_emp_p][lvl_id_1+1:lvl_id_2+1, :, :], dims=1), dims=3))
        yY = vec(mean(sum(all_temp_level_dict[t][:all_dkl_per_lev_emp_p][:, :, :], dims=1), dims=3))
        cnow = get_c(cc3, allttsminorforc, t)

        l1l, = ax.plot(Mss[mss_id], y[mss_id], linestyle="-", c="w", lw=1.5, zorder=-50,
            path_effects=[pe.withStroke(linewidth=3., foreground="darkolivegreen"), pe.Normal()], solid_capstyle="butt")
        l2l, = ax.plot(Mss[mss_id], y[mss_id], color=cnow, zorder=-20, linewidth=1.5, solid_capstyle="butt", ls="--")
        l3l, = axin.plot(Mss[mss_id], yY[mss_id], linestyle="-", c=cnow, lw=0.75, zorder=20,
            path_effects=[pe.withStroke(linewidth=2, foreground="darkolivegreen"), pe.Normal()], solid_capstyle="butt")

        y2  = vec(mean(sum(all_temp_level_dict[t][:all_dkl_per_lev_q_p][lvl_id_1+1:lvl_id_2+1, :, :], dims=1), dims=3))
        yY2 = vec(mean(sum(all_temp_level_dict[t][:all_dkl_per_lev_q_p][:, :, :], dims=1), dims=3))

        l4l, = ax.plot(Mss[mss_id], y2[mss_id], linestyle="-", c="w", lw=1.5, zorder=-150,
            path_effects=[pe.withStroke(linewidth=3, foreground="steelblue"), pe.Normal()], solid_capstyle="butt")
        l5l, = ax.plot(Mss[mss_id], y2[mss_id], color=cnow, zorder=-100, linewidth=1.5, solid_capstyle="butt", ls="--")
        l6l, = axin.plot(Mss[mss_id], yY2[mss_id], linestyle="-", c=cnow, lw=0.75, zorder=-200,
            path_effects=[pe.withStroke(linewidth=2, foreground="steelblue"), pe.Normal()], solid_capstyle="butt")
        axin.set_xscale("log")

        if iil == 2
            ax.legend(((l1l, l2l), (l4l, l5l)),
                (L"\langle D_{KL}( p_{\mathcal{D}} || p_T  ) \cap \Lambda_4 \rangle_{\text{reps}}",
                 L"\langle D_{KL}(\hat{q} || p_T) \cap  \Lambda_4\rangle_{\text{reps}}"),
                loc=legloc, frameon=false)
            axin.legend(((l3l,), (l6l,)),
                (L"\langle D_{KL}( p_{\mathcal{D}} || p_T )\rangle_{\text{reps}}",
                 L"\langle D_{KL}(\hat{q} || p_T)\rangle_{\text{reps}}"),
                loc=axinlegloc, frameon=false, handletextpad=0.1, handlelength=1.5, labelspacing=0.01)
        end
    end
    axin.set_ylim(-0.5, 6.7)

    ax.set_xscale("log")
    ax.set_xlabel("training data size " * L"M")
    ax.set_ylabel("nats")

    cax = ax.inset_axes(cb_loc)
    cbarr = fig.colorbar(matplotlib.cm.ScalarMappable(cmap="Greys",
                norm=matplotlib.colors.Normalize(vmin=minimum(allttsminorforc), vmax=maximum(allttsminorforc))),
        cax=cax, orientation="horizontal", label=L"T", pad=0.05)
    cbarr.ax.xaxis.set_ticks_position("bottom")
    cbarr.ax.xaxis.set_label_position("top")
    return nothing
end

In [ ]:
# Load the committed summary dicts (in this folder).
hld = jldopen(f -> f["data"], joinpath(@__DIR__, "hld_dict_for_last_figs.jld2"), "r")
all_temp_level_dict = jldopen(f -> f["data"], joinpath(@__DIR__, "all_temp_level_dict.jld2"), "r")

# Temperatures present, and the training-set-size grid (matches dim 2 of the arrays = 13).
all_tts = sort(collect(keys(all_temp_level_dict)))     # [2.3, 2.4, 2.6, 2.8, 3.2, 3.6, 4.0, 4.4]
Mss     = Int.(round.(10 .^ collect(1:0.2:4)))[4:end]  # 13 values, aligns with all_dkl_per_lev_* dim 2

@show hld[:low][:temp] hld[:high][:temp] all_tts length(Mss);

In [ ]:
# Reproduce panels a–d of appendix figure `nn_ising_kappa_C.pdf` (Appendix C).
fig, ax = subplots(2, 2, sharex=true, dpi=300, figsize=set_size_for_paper(1, 1.))
plot_dkl_et_al(ax, hld)

mkpath(plotsdir("supplemental"))
outpath = plotsdir("supplemental", "nn_ising_kappa_C_ad_reproduced.svg")
savefig(outpath, bbox_inches="tight")
println("saved reproduced panels a–d to: ", outpath)
fig

In [ ]:
# Reproduce `supp_fig_working-new.pdf` (Appendix D).
fig, ax = subplots(figsize=set_size_for_paper(1, 1.), dpi=300)
fig4_3(ax; inset_loc=(0.6, 0.6, 0.3, 0.3), cb_loc=[0.25, 0.9, 0.27, 0.025],
       axinlegloc=(-1.5, 0.15), legloc=(0.4, 0.42))
ax.set_ylim(-0.05, 1.5)

mkpath(plotsdir("supplemental"))
outpath = plotsdir("supplemental", "supp_fig_reproduced.svg")
savefig(outpath, bbox_inches="tight")
println("saved reproduced supp_fig to: ", outpath)
fig